# Notebook 4: Survey Analysis (110 small hospitality business owners)
This notebook analyzes your survey (Google Forms export). It gives you:
1. **Who answered** (profile of respondents)
2. **How they set prices now** (B1-B5)
3. **Technology and AI attitudes** (C1-C5)
4. **Cross-checks** (does confidence relate to willingness to use AI?)
5. **Open answers** (D1 and D2) with a simple theme count
6. **Requirements table** (what the survey says the system must do)

**Before you run:** upload the Google Form CSV to `pricing_project/data/` and rename it to `survey.csv`.

## Step 1: Load and clean

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/pricing_project'
except ImportError:
    BASE = os.environ.get('PROJECT_DIR', '.')

OUT = f'{BASE}/results/survey'
os.makedirs(OUT, exist_ok=True)

raw = pd.read_csv(f'{BASE}/data/survey.csv')
raw.columns = [c.strip() for c in raw.columns]

# Short names so the code is easy to read
cols = list(raw.columns)
short = ['time', 'age', 'gender', 'education', 'A1_experience', 'A2_type', 'A3_rooms', 'A4_years', 'A5_district',
         'B1_how_price', 'B2_frequency', 'B3_factors', 'B4_maximise', 'B5_challenge',
         'C1_platforms', 'C2_confidence', 'C3_ai_aware', 'C4_would_use', 'C5_trust', 'D1_difficult', 'D2_other']
df = raw.copy(); df.columns = short
df['time'] = pd.to_datetime(df['time'])

# Clean district names (extra spaces, small letters, one invalid answer)
df['A5_district'] = df.A5_district.str.strip().str.title()
df.loc[df.A5_district.str.len() > 20, 'A5_district'] = 'Invalid answer'

print('Respondents:', len(df))
print('Collected from', df.time.min().date(), 'to', df.time.max().date())
print('Duplicate rows:', int(df.drop(columns='time').duplicated().sum()))
print('Missing per column:'); print(df.isna().sum()[df.isna().sum() > 0])

**Data quality notes** (write them in your Limitations):
- One answer in A2 is "nothing at all" (not a valid business type).
- Several open answers (D1) are word-for-word the same for different people. Check if this is normal (copy-paste, shared answers) and mention it.

In [ ]:
print(df.A2_type.value_counts())
print()
print('Identical D1 answers repeated:', int(df.D1_difficult.dropna().duplicated().sum()), 'of', int(df.D1_difficult.notna().sum()))

## Step 2: Helper functions

In [ ]:
def pct_table(series, order=None):
    t = series.value_counts().rename('count').to_frame()
    t['percent'] = (t['count'] / series.notna().sum() * 100).round(1)
    if order: t = t.reindex(order).fillna(0)
    return t

def multi_table(series, options):
    # count how many people ticked each option (options are matched by their first words)
    rows = []
    n = series.notna().sum()
    for key, label in options.items():
        c = series.fillna('').str.contains(re.escape(key), regex=True).sum()
        rows.append({'option': label, 'count': c, 'percent': round(c / n * 100, 1)})
    return pd.DataFrame(rows).sort_values('count', ascending=False).reset_index(drop=True)

def bar(table, label_col, title, fname):
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(t))))
    ax.barh(t[label_col].astype(str).str.slice(0, 70), t['percent'], color='#2e7d32')
    for i, v in enumerate(t['percent']):
        ax.text(v + 0.5, i, f'{v}%', va='center')
    ax.set_xlabel('% of respondents'); ax.set_title(title); ax.set_xlim(0, max(t['percent']) * 1.15)
    plt.tight_layout(); plt.savefig(f'{OUT}/{fname}.png', dpi=150); plt.show()

def save(t, name): t.to_csv(f'{OUT}/{name}.csv')

## Step 3: Who answered? (profile)

In [ ]:
for col in ['age', 'gender', 'education', 'A1_experience', 'A2_type', 'A3_rooms', 'A4_years']:
    t = pct_table(df[col]); save(t, f'profile_{col}')
    print('\n' + col); print(t)

t = pct_table(df.A5_district); save(t, 'profile_district'); print('\nDistricts:'); print(t)

## Step 4: How do owners set prices now? (B1-B5)

In [ ]:
B1_opts = {
    'I adjust prices based on the season': 'Adjust by season (peak / off-peak)',
    'I check competitor prices': 'Check and match competitor prices',
    'I base it on my own experience': 'Own experience and judgment',
    'I set a fixed price': 'Fixed price that rarely changes',
    'I use a pricing tool': 'Use a pricing tool / software',
}
t = multi_table(df.B1_how_price, B1_opts); save(t, 'B1_how_price'); print(t); bar(t, 'option', 'B1: How owners decide room price', 'B1')

In [ ]:
order = ['Daily or in real time', 'Monthly', 'Once or twice a year (seasonal change only)', 'Never — I keep one fixed price']
t = pct_table(df.B2_frequency, order); save(t, 'B2_frequency'); print(t); bar(t.reset_index(names='option'), 'option', 'B2: How often prices change', 'B2')

In [ ]:
B3_opts = {
    'Tourist arrival season': 'Tourist arrival season',
    'My running costs': 'Own running costs',
    'Competitor prices': 'Competitor prices',
    'My own occupancy': 'Own occupancy rate',
    'Local events': 'Local events / festivals',
    'Weather': 'Weather / climate',
    'Customer reviews': 'Customer reviews',
    'I do not consider': 'No specific factor',
}
t = multi_table(df.B3_factors, B3_opts); save(t, 'B3_factors'); print(t); bar(t, 'option', 'B3: Factors used when setting price', 'B3')

In [ ]:
t = pct_table(df.B4_maximise); save(t, 'B4_maximise'); print(t)
t = pct_table(df.B5_challenge); save(t, 'B5_challenge'); print(t)
bar(t.reset_index(names='option'), 'option', 'B5: Biggest challenge when setting prices', 'B5')

## Step 5: Technology and AI (C1-C5)

In [ ]:
C1_opts = {'Booking.com': 'Booking.com', 'Airbnb': 'Airbnb', 'Agoda': 'Agoda', 'My own website': 'Own website',
           'TripAdvisor': 'TripAdvisor', 'WhatsApp or phone': 'WhatsApp / phone only', 'I do not use any': 'No online platform'}
t = multi_table(df.C1_platforms, C1_opts); save(t, 'C1_platforms'); print(t)

t = pct_table(df.C2_confidence); save(t, 'C2_confidence'); print(t)
t = pct_table(df.C3_ai_aware); save(t, 'C3_ai_aware'); print(t)
t = pct_table(df.C4_would_use); save(t, 'C4_would_use'); print(t)
bar(t.reset_index(names='option'), 'option', 'C4: Would you use an AI pricing tool?', 'C4')

In [ ]:
C5_opts = {
    'If it showed me the data': 'Shows the data used',
    'If it explained why': 'Explains why each price',
    'If other hotel owners': 'Other owners use and recommend it',
    'If there was a tutorial': 'Built-in tutorial / guide',
    'If I could override': 'I can change the price myself',
    'Nothing': 'Nothing - would not use',
}
t = multi_table(df.C5_trust, C5_opts); save(t, 'C5_trust'); print(t); bar(t, 'option', 'C5: What would make you trust an AI pricing tool', 'C5')

## Step 6: Cross-checks

In [ ]:
from scipy.stats import fisher_exact

confident = df.C2_confidence.str.startswith(('Very', 'Somewhat'))
would_use = df.C4_would_use.str.startswith(('Yes', 'Maybe'))
ct = pd.crosstab(confident.map({True: 'Confident with technology', False: 'Not confident'}),
                 would_use.map({True: 'Would use (yes / maybe)', False: 'Would not use'}))
print(ct); print((ct.div(ct.sum(axis=1), axis=0) * 100).round(1))
odds, p = fisher_exact(ct.values)
print(f'\nFisher exact test p-value = {p:.3f}  ({"significant" if p < 0.05 else "not significant"} at 0.05)')
save(ct, 'cross_confidence_vs_use')

print()
ct2 = pd.crosstab(df.A1_experience, df.B2_frequency); print(ct2); save(ct2, 'cross_experience_vs_frequency')
print()
ct3 = pd.crosstab(df.C3_ai_aware, df.C4_would_use); print(ct3); save(ct3, 'cross_awareness_vs_use')

## Step 7: Open answers (D1 and D2) - simple theme count
This is a **keyword helper**, not the final analysis. Read the answers yourself, then adjust the keywords. For your thesis, say you used thematic analysis (Braun & Clarke, 2006), supported by keyword counts.

In [ ]:
themes_d1 = {
    'Competitor prices / price wars': r'competit|price war|undercut|what .* charging|nearby .* charg|too much supply|to much supply|so many|new guest houses',
    'Demand, season and events are hard to predict': r'predict|season|monsoon|off.?season|peak|festival|perahera|poya|pilgrim|holiday|cricket|event|conference|forecast',
    'Weather, travel disruptions, cancellations': r'weather|rain|cancel|no-?show|flight|train|strike|delay|rough sea',
    'High running costs': r'electric|maintain|maintainance|tariff|bill|cost|petrol|food price|staff',
    'Different guest types / bargaining': r'local .*foreign|foreign .*local|backpack|bargain|russian|european|transit|corporate|ayurvedic',
    'Time and managing many platforms': r'no time|dont have time|not having enough time|manually|double booking|every week|every day|one person|cannot monitor',
    'Not enough data / guesswork': r'guess|dont have .*idea|quite idea|know how much|willing to pay|enough review|dont no|quality information',
}
themes_d2 = {
    'Wants automated price recommendation': r'automated|recommend|fair price|suggest',
    'Sinhala / Tamil language and local training': r'sinhala|tamil|training|youtube|learn|guide',
    'Simple and mobile-friendly': r'simple|mobile|phone|whatsapp|sms|offline|easy',
    'Cost (tool price or running cost)': r'\bcost\b|free|low cost|subscription',
    'Wants explanations and trust in data': r'explain|trust|sltda|official',
    'Wants control (override price)': r'override|control',
    'Platform commission and fees': r'commission|flatform|freez|undercut',
    'Wants weather / events / festivals in the tool': r'weather|festival|poya|event|forecast|prediction',
}

def theme_count(series, themes):
    s = series.dropna().astype(str)
    s = s[s.str.len() > 6]                  # remove empty answers like "Nil", "No"
    rows = [{'theme': k, 'count': int(s.str.contains(v, case=False, regex=True).sum())} for k, v in themes.items()]
    t = pd.DataFrame(rows).sort_values('count', ascending=False).reset_index(drop=True)
    t['percent_of_answers'] = (t['count'] / len(s) * 100).round(1)
    return t, len(s)

t1, n1 = theme_count(df.D1_difficult, themes_d1); print(f'D1: {n1} usable answers'); print(t1); save(t1, 'D1_themes')
t2, n2 = theme_count(df.D2_other, themes_d2);     print(f'\nD2: {n2} usable answers'); print(t2); save(t2, 'D2_themes')

In [ ]:
# Look at 2-3 answers per theme to check the keywords are correct
for k, v in themes_d1.items():
    ex = df.D1_difficult.dropna()[df.D1_difficult.dropna().str.contains(v, case=False, regex=True)].drop_duplicates().head(2)
    print('\n#', k)
    for e in ex: print('  -', e[:150])

## Step 8: What the survey means for the system (requirements)

In [ ]:
def p(series, opts, key):  # percent helper
    t = multi_table(series, opts); return float(t.loc[t.option == opts[key], 'percent'].iloc[0])

req = pd.DataFrame([
    ['Show the data used for each price', f"{p(df.C5_trust, C5_opts, 'If it showed me the data')}% said this builds trust", 'Show real values (arrivals, season, last week) next to the normal values'],
    ['Explain why each price was suggested', f"{p(df.C5_trust, C5_opts, 'If it explained why')}% said this builds trust", 'SHAP-based "Why this price?" screen'],
    ['Let the owner change the price', f"{p(df.C5_trust, C5_opts, 'If I could override')}% asked for this", 'Owner can accept or change each suggested price (future work if not built)'],
    ['Built-in guide / tutorial', f"{p(df.C5_trust, C5_opts, 'If there was a tutorial')}% asked for this", 'Short "How it works" guide'],
    ['Use season and competitor prices', f"{p(df.B3_factors, B3_opts, 'Tourist arrival season')}% use season; {p(df.B3_factors, B3_opts, 'Competitor prices')}% use competitor prices", 'Demand forecast from SLTDA data + competitor prices'],
    ['Sinhala / Tamil and mobile friendly', 'Raised in open answers (D2)', 'Three languages, mobile-first design'],
], columns=['Requirement', 'Survey evidence', 'How the system answers it'])
save(req.set_index('Requirement'), 'requirements_from_survey'); req

## Done. Files are in `results/survey/` (tables as CSV, charts as PNG).
Use these in **Chapter 3 (requirement gathering)** and **Chapter 5 (survey results)**.